In [1]:
from networkx.algorithms import bipartite
from copy import deepcopy
from scipy import stats
from scipy.spatial import cKDTree
import pandas as pd
import os
import numpy as np
from tqdm import tqdm
import networkx as nx
import random
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Union, Set

seed = 0
# Set the random seed for reproducibility
random.seed(seed)
np.random.seed(seed)

base_sparsified_path = r'/raid/t2/TGN_adv/sparsified_data'
dataset_name = 'wikipedia'

original_filename = rf'ml_{dataset_name}.csv'

print(os.path.join(base_sparsified_path, dataset_name, original_filename))

orig_df = pd.read_csv(os.path.join(base_sparsified_path, dataset_name, original_filename))
orig_df.drop(['Unnamed: 0'], axis=1, inplace=True)
orig_df.head()

/raid/t2/TGN_adv/sparsified_data/wikipedia/ml_wikipedia.csv


,u,i,ts,label,idx
0,1,8228,0.0,0.0,1
1,2,8229,36.0,0.0,2
2,2,8229,77.0,0.0,3
3,3,8230,131.0,0.0,4
4,2,8229,150.0,0.0,5


In [2]:
# sparsification 
sprs_type = 'ts_tpr_remove_MSS'
upto = 0.7

sprs_filename = f'{dataset_name}_{sprs_type}_sparsified_{upto}.csv'
sprs_filename

full_sprs_filepath = os.path.join(base_sparsified_path, dataset_name, sprs_type, sprs_filename)

sprs_df = pd.read_csv(full_sprs_filepath)
sprs_df.drop(['Unnamed: 0', 'Unnamed: 0.1'], axis=1, inplace=True)
sprs_df.head()

,u,i,ts,label,idx
0,1,8228,0.0,0.0,1
1,2181,8735,429397.0,0.0,20857
2,2198,8735,433126.0,0.0,21023
3,2206,8536,434533.0,0.0,21107
4,2207,8540,434578.0,0.0,21110


In [3]:
def get_nodes_in_time_window(df: pd.DataFrame, current_time: float, window: int) -> Set[int]:
    """
    Get the set of nodes that were active within the time window before current_time.
    
    Args:
    df (pd.DataFrame): DataFrame with interactions ('u' for source, 'i' for destination, 'ts' for timestamp)
    current_time (float): Current timestamp
    window (int): Time window size
    
    Returns:
    Set[int]: Set of nodes active within the time window
    """
    window_start = max(current_time - window, df['ts'].min())
    window_df = df[(df['ts'] >= window_start) & (df['ts'] <= current_time)]
    return set(window_df['u']) | set(window_df['i'])

class TemporalSampler:
    def __init__(self, original_graph, seed=None, time_window=1200):
        self.original_graph = original_graph
        self.seed=seed
        self.rng = np.random.default_rng(seed)
        self.time_window = time_window
        self.min_time = min(0, original_graph['ts'].min())
        self.max_time = original_graph['ts'].max()
        self.initialize_kde()
        self.initialize_kd_tree()

    def initialize_kde(self):
        df_sorted = self.original_graph.sort_values('ts')
        inter_event_times = np.diff(df_sorted['ts'])
        self.kde = stats.gaussian_kde(inter_event_times)
        # Set the random state for KDE
        # self.kde.set_bandwidth(bw_method=self.kde.factor)
        # self.kde.reseed(self.seed)

    def initialize_kd_tree(self):
        self.temporal_kd_tree = cKDTree(self.original_graph['ts'].values.reshape(-1, 1))

    def assign_timestamps(self, num_samples):
        kde_samples = self.kde.resample(num_samples, self.rng)[0]
        cumulative_times = np.cumsum(kde_samples) + self.min_time
        cumulative_times = np.clip(cumulative_times, self.min_time, self.max_time)
        jitter = self.rng.uniform(0, self.time_window * 0.1, num_samples)
        sampled_timestamps = cumulative_times + jitter
        sampled_timestamps.sort()
        sampled_timestamps = np.clip(sampled_timestamps, self.min_time, self.max_time)
        return sampled_timestamps

    def find_next_events(self, timestamps):
        _, indices = self.temporal_kd_tree.query(timestamps.reshape(-1, 1), k=1)
        return self.original_graph['ts'].values[indices]

def create_node_mapping(original_degrees: Dict[int, int], generated_degrees: Dict[int, int]) -> Dict[int, int]:
    """
    Create a one-to-one mapping between generated and original nodes based on degree.
    
    Args:
    original_degrees (Dict[int, int]): Degree distribution of original nodes
    generated_degrees (Dict[int, int]): Degree distribution of generated nodes
    
    Returns:
    Dict[int, int]: Mapping from generated nodes to original nodes
    """
    degree_to_nodes = defaultdict(list)
    for node, degree in original_degrees.items():
        degree_to_nodes[degree].append(node)
    
    generated_degree_to_nodes = defaultdict(list)
    for node, degree in generated_degrees.items():
        generated_degree_to_nodes[degree].append(node)
    
    mapping = {}
    for degree, nodes in degree_to_nodes.items():
        generated_nodes = generated_degree_to_nodes.get(degree, [])
        for original_node, generated_node in zip(nodes, generated_nodes):
            mapping[generated_node] = original_node
    
    return mapping

In [4]:
early_start_time, early_end_time = sprs_df['ts'].iloc[0], sprs_df['ts'].iloc[2]
early_df = orig_df[(orig_df['ts'] > early_start_time )& (orig_df['ts'] < early_end_time)]
src_nodes_degrees = Counter(early_df['u'])
dst_nodes_degrees = Counter(early_df['i'])

In [5]:
generated_simple_graphs = nx.bipartite.havel_hakimi_graph(aseq=list(src_nodes_degrees.values()),
                                                          bseq=list(dst_nodes_degrees.values()),
                                                          create_using=nx.Graph())
generated_edges = generated_simple_graphs.edges()

In [6]:
# Initialize TemporalSampler
sampler = TemporalSampler(early_df, time_window=1200, seed=seed)

# Generate timestamps for all potential negative samples
num_samples = len(generated_edges)
sampled_timestamps = sampler.assign_timestamps(num_samples)
len(generated_edges), len(sampled_timestamps)

(21021, 21021)

In [7]:
n = len(src_nodes_degrees)
degrees = dict(generated_simple_graphs.degree())
degrees_top = {k: degrees[k] for k in list(degrees.keys())[:n]}
degrees_bottom = {k: degrees[k] for k in list(degrees.keys())[n:]}

In [8]:
src_mapping = create_node_mapping(src_nodes_degrees, degrees_top)
dst_mapping = create_node_mapping(dst_nodes_degrees, degrees_bottom)

len(src_mapping), len(dst_mapping)

(2196, 656)

In [9]:
print(orig_df['u'].min(), orig_df['u'].max())
print(orig_df['i'].min(), orig_df['i'].max())

1 8227
8228 9227


In [10]:
from itertools import product

df_sorted =early_df.sort_values('ts')
original_edges = set(tuple(x) for x in early_df[['u', 'i']].values)

min_src, max_src = orig_df['u'].min(), orig_df['u'].max()
min_dst, max_dst = orig_df['i'].min(), orig_df['i'].max()

map_ts_to_edges = {}

# map_edges_to_ts = {}
map_edges_to_ts = defaultdict(list)

for timestamp in tqdm(sampled_timestamps, desc='Processing'):
    nodes_in_window = get_nodes_in_time_window(df_sorted, timestamp, 1200)
    
    src_nodes_orig = [n for n in nodes_in_window if min_src <= n <= max_src]  
    dst_nodes_orig = [n for n in nodes_in_window if min_dst <= n < max_dst]
    
    edges = [edge
        for edge in product(src_nodes_orig, dst_nodes_orig)
        if edge not in original_edges
    ]
    
    # edges = [
    #     (u, v) for u in src_nodes_orig for v in dst_nodes_orig
    #     if (u, v) not in original_edges
    # ]
    
    # assert len(edges) > 0, f'No such edge for {timestamp = }'
    
    map_ts_to_edges[timestamp] = edges
    for edge in edges:
        if edge not in map_edges_to_ts.keys():
            map_edges_to_ts[edge] = []
        map_edges_to_ts[edge].append(timestamp)
    

Processing: 100%|██████████| 21021/21021 [00:17<00:00, 1185.34it/s]


In [11]:
# from collections import defaultdict
# from tqdm import tqdm

# # Preprocess for efficiency: convert edges to a set for fast lookup
# existing_edges_set = set(tuple(x) for x in early_df[['u', 'i']].values)

# # Use defaultdict for map_edges_to_ts to avoid checking for keys
# map_edges_to_ts = defaultdict(list)

# # Calculate min/max bounds once before the loop
# u_min, u_max = orig_df['u'].min(), orig_df['u'].max()
# i_min, i_max = orig_df['i'].min(), orig_df['i'].max()

# # Process each timestamp
# for timestamp in tqdm(sampled_timestamps, desc='Processing'):
#     nodes_in_window = get_nodes_in_time_window(df_sorted, timestamp, 1200)
    
#     # Filter nodes once based on the pre-calculated bounds
#     src_nodes_orig = [n for n in nodes_in_window if u_min <= n <= u_max]
#     dst_nodes_orig = [n for n in nodes_in_window if i_min <= n < i_max]

#     # Generate valid edges while checking for existence in the set
#     edges = [
#         (u, v) for u in src_nodes_orig for v in dst_nodes_orig
#         # if (u, v) not in existing_edges_set
#     ]
#     # and (u, v) in generated_edges
#     assert len(edges) > 0, f'No such edge for {timestamp = }'
    
#     # Store edges for the current timestamp
#     map_ts_to_edges[timestamp] = edges

#     # Update the map_edges_to_ts mapping
#     for edge in edges:
#         map_edges_to_ts[edge].append(timestamp)


In [12]:
len(map_edges_to_ts.keys())

241593

In [13]:
feasible_edges_dict = deepcopy(map_edges_to_ts)
# sampled_timestamps

gen_edges = [(src_mapping[edge[0]], dst_mapping[edge[1]]) for edge in generated_edges]

degree_dict = deepcopy(src_nodes_degrees)
degree_dict.update(dst_nodes_degrees)

In [ ]:
# Old Code

def assign_edges_to_timestamps(edges, timestamps, feasible_edges_dict, degree_dict, degree_limit):
    neg_samples = []
    pos_samples = []
    # Initialize tracking structures
    last_active_timestamp = {node: -float('inf') for node in degree_dict.keys()}
    assigned_edges_upto_timestamp = {t: 0 for t in timestamps}
    node_degree_remaining = {node: {t: degree_limit[node] for t in timestamps} for node in degree_dict.keys()}
    
    # Sort edges based on node degree (edge prioritization)
    edges.sort(key=lambda e: max(degree_dict[e[0]], degree_dict[e[1]]), reverse=True)
    
    # Loop through prioritized edges
    for (u, v) in edges:
        # Step 1: Get feasible timestamps for edge (u, v) from feasible_edges_dict
        feasible_timestamps = feasible_edges_dict[(u, v)]
        
        # Step 2: Select best timestamp based on timestamp utilization score
        if feasible_timestamps:
            best_timestamp = max(feasible_timestamps, key=lambda t: timestamp_utilization_score(u, v, t, assigned_edges_upto_timestamp, node_degree_remaining))
            
            # Step 3: Assign edge to best_timestamp and update the state
            assigned_edges_upto_timestamp[best_timestamp] += 1
            last_active_timestamp[u] = best_timestamp
            last_active_timestamp[v] = best_timestamp
            node_degree_remaining[u][best_timestamp] -= 1
            node_degree_remaining[v][best_timestamp] -= 1
            
            # print(f"Assigned edge ({u}, {v}) to timestamp {best_timestamp}")
            neg_samples.append((u, v, best_timestamp))
        else:
            # print(f"Edge ({u}, {v}) could not be assigned to any feasible timestamp.")
            pos_samples.append((u, v, None))
    return neg_samples, pos_samples
    
def timestamp_utilization_score(u, v, t, assigned_edges_upto_timestamp, node_degree_remaining):
    # Higher score means better timestamp
    return -assigned_edges_upto_timestamp[t] + node_degree_remaining[u][t] + node_degree_remaining[v][t]


In [ ]:
def assign_edges_to_timestamps(edges, timestamps, feasible_edges_dict, degree_dict, degree_limit):
    neg_samples = []
    pos_samples = []
    
    # Enhanced tracking structures
    last_active_timestamp = {node: -float('inf') for node in degree_dict.keys()}
    assigned_edges_upto_timestamp = {t: 0 for t in timestamps}
    node_degree_remaining = {node: {t: degree_limit[node] for t in timestamps} 
                           for node in degree_dict.keys()}
    
    # New tracking for reuse phase
    timestamp_capacity = {t: 5 for t in timestamps}  # Max edges per timestamp, set to 5 rn
    edge_assignments = {t: [] for t in timestamps}  # Track which edges assigned where
    edge_usage_count = {(u, v): 0 for u, v in edges}  # Track how often each edge used
    
    # Phase 1: Initial Assignment
    edges.sort(key=lambda e: max(degree_dict[e[0]], degree_dict[e[1]]), reverse=True)
    for (u, v) in edges:
        feasible_timestamps = feasible_edges_dict[(u, v)]
        if feasible_timestamps:
            best_timestamp = max(feasible_timestamps, 
                               key=lambda t: timestamp_utilization_score(u, v, t, 
                                           assigned_edges_upto_timestamp, 
                                           node_degree_remaining))
            
            # Assign edge and update tracking
            assigned_edges_upto_timestamp[best_timestamp] += 1
            last_active_timestamp[u] = best_timestamp
            last_active_timestamp[v] = best_timestamp
            node_degree_remaining[u][best_timestamp] -= 1
            node_degree_remaining[v][best_timestamp] -= 1
            edge_assignments[best_timestamp].append((u, v))
            edge_usage_count[(u, v)] += 1
            neg_samples.append((u, v, best_timestamp))
        else:
            pos_samples.append((u, v, None))
    
    # Phase 2: Fill underutilized timestamps
    for t in sorted(timestamps, 
                   key=lambda x: timestamp_capacity[x] - assigned_edges_upto_timestamp[x],
                   reverse=True):
        remaining_capacity = timestamp_capacity[t] - assigned_edges_upto_timestamp[t]
        if remaining_capacity <= 0:
            continue
            
        # Find eligible edges for reuse at this timestamp
        eligible_edges = []
        for (u, v) in edges:
            if t in feasible_edges_dict[(u, v)] and \
               node_degree_remaining[u][t] > 0 and \
               node_degree_remaining[v][t] > 0:
                # Score this edge for reuse
                reuse_score = (
                    len(feasible_edges_dict[(u, v)]),  # More feasible timestamps = better
                    node_degree_remaining[u][t] + node_degree_remaining[v][t]  # More remaining degree = better
                )
                eligible_edges.append(((u, v), reuse_score))
        
        # Sort eligible edges by reuse score
        eligible_edges.sort(key=lambda x: x[1], reverse=True)
        
        # Assign edges until capacity filled or no more eligible edges
        for (u, v), _ in eligible_edges:
            if assigned_edges_upto_timestamp[t] >= timestamp_capacity[t]:
                break
                
            # Reuse this edge
            assigned_edges_upto_timestamp[t] += 1
            node_degree_remaining[u][t] -= 1
            node_degree_remaining[v][t] -= 1
            edge_assignments[t].append((u, v))
            edge_usage_count[(u, v)] += 1
            neg_samples.append((u, v, t))
            
            if assigned_edges_upto_timestamp[t] >= timestamp_capacity[t]:
                break
    
    return neg_samples, pos_samples

def timestamp_utilization_score(u, v, t, assigned_edges_upto_timestamp, node_degree_remaining):
    """Score a timestamp for edge assignment based on utilization and remaining capacity"""
    current_utilization = assigned_edges_upto_timestamp[t]
    degree_availability = min(node_degree_remaining[u][t], node_degree_remaining[v][t])
    return -(current_utilization * 100 + degree_availability)  # Negative to prefer less utilized timestamps

In [ ]:
# V3
def assign_edges_to_timestamps(edges, timestamps, feasible_edges_dict, degree_dict, degree_limit):
    neg_samples = []
    pos_samples = []
    
    all_keys = list(feasible_edges_dict.keys())
    
    # Initialize tracking structures
    node_degree_remaining = {node: {t: degree_limit[node] for t in timestamps} 
                           for node in degree_dict.keys()}
    assigned_edges_upto_timestamp = {t: 0 for t in timestamps}
    
    # Phase 1: Initial Assignment
    edges.sort(key=lambda e: max(degree_dict[e[0]], degree_dict[e[1]]), reverse=True)
    
    neg_samples, pos_samples = get_timestaps(edges, feasible_edges_dict, neg_samples, pos_samples, node_degree_remaining, assigned_edges_upto_timestamp)
            
    for (u, v, _) in pos_samples:
        possible_repl_u =list([(k, _v) for (k, _v) in all_keys if (u == k)])
        possible_repl_v = list([(k, _v) for (k, _v) in all_keys if (v == _v)])
        
        
        assert len(possible_repl_u) > 1, f"Not possible to find replacement for {(u, v)}"
        assert len(possible_repl_v) > 1, f"Not possible to find replacement for {(u, v)}"
        
        edge_u, edge_v = random.choice(possible_repl_u), random.choice(possible_repl_v)
        
        # print(edge_u, edge_v)
        
        # assert 0 == 1, 'break'
        
        new_neg_samples, new_pos_samples = get_timestaps([edge_u, edge_v], feasible_edges_dict, neg_samples, pos_samples,
                                                 node_degree_remaining, assigned_edges_upto_timestamp)
        # There would be some issue because of this, but I cant wrap my head around it.
        
        
    
    return new_neg_samples, new_pos_samples

def get_timestaps(edges, feasible_edges_dict, neg_samples, pos_samples, node_degree_remaining, assigned_edges_upto_timestamp):
    for (u, v) in edges:
        feasible_timestamps = feasible_edges_dict[(u, v)] 
        # check if edge exist in keys of any edge dict if not find two edges (u, v') and (u', v) in feasible_edges_dict.keys()
        # find their feasible timestamps and chose best timestamps
        # Update everything
        if feasible_timestamps:
            best_timestamp = max(feasible_timestamps, 
                               key=lambda t: timestamp_utilization_score(u, v, t, 
                                           assigned_edges_upto_timestamp, 
                                           node_degree_remaining))
            
            # Assign edge and update tracking
            assigned_edges_upto_timestamp[best_timestamp] += 1
            node_degree_remaining[u][best_timestamp] -= 1
            node_degree_remaining[v][best_timestamp] -= 1
            neg_samples.append((u, v, best_timestamp))
        else:
            pos_samples.append((u, v, None))
    return neg_samples, pos_samples

def timestamp_utilization_score(u, v, t, assigned_edges_upto_timestamp, node_degree_remaining):
    """Score a timestamp for edge assignment based on utilization and remaining capacity"""
    current_utilization = assigned_edges_upto_timestamp[t]
    degree_availability = min(node_degree_remaining[u][t], node_degree_remaining[v][t])
    return -(current_utilization * 100 + degree_availability)

In [15]:
# V4

def assign_edges_to_timestamps(edges, timestamps, feasible_edges_dict, degree_dict, degree_limit):
    neg_samples = []
    pos_samples = []
    
    all_keys = list(feasible_edges_dict.keys())
    
    # Initialize tracking structures
    node_degree_remaining = {node: {t: degree_limit[node] for t in timestamps} 
                             for node in degree_dict.keys()}
    assigned_edges_upto_timestamp = {t: 0 for t in timestamps}
    
    # Phase 1: Initial Assignment
    edges.sort(key=lambda e: max(degree_dict[e[0]], degree_dict[e[1]]), reverse=True)
    
    neg_samples, pos_samples = get_timestaps(edges, feasible_edges_dict, neg_samples, pos_samples, 
                                             node_degree_remaining, assigned_edges_upto_timestamp)
    
    for (u, v, _) in pos_samples:
        possible_repl_u = list([(k, _v) for (k, _v) in all_keys if (u == k)])
        possible_repl_v = list([(k, _v) for (k, _v) in all_keys if (v == _v)])
        
        assert len(possible_repl_u) > 1, f"Not possible to find replacement for {(u, v)}"
        assert len(possible_repl_v) > 1, f"Not possible to find replacement for {(u, v)}"
        
        edge_u, edge_v = random.choice(possible_repl_u), random.choice(possible_repl_v)
        
        # Reassign edges and update
        # we are not repeating older negative samples as (u,v) might not reflect true node degree
        new_neg_samples, new_pos_samples = get_timestaps([edge_u, edge_v], feasible_edges_dict, 
                                                         neg_samples, pos_samples,
                                                         node_degree_remaining, 
                                                         assigned_edges_upto_timestamp)
        
    return new_neg_samples, new_pos_samples

def get_timestaps(edges, feasible_edges_dict, neg_samples, pos_samples, node_degree_remaining, 
                  assigned_edges_upto_timestamp):
    for (u, v) in edges:
        feasible_timestamps = feasible_edges_dict[(u, v)] 
        
        if feasible_timestamps:
            # Step 1: Filter timestamps where node degree limits aren't exceeded
            feasible_timestamps = [t for t in feasible_timestamps 
                                   if node_degree_remaining[u][t] > 0 and node_degree_remaining[v][t] > 0]
            
            if feasible_timestamps:
                # Step 2: Select the best timestamp based on utilization score
                best_timestamp = max(feasible_timestamps, key=lambda t: timestamp_utilization_score(
                    u, v, t, assigned_edges_upto_timestamp, node_degree_remaining))
                
                # Step 3: Assign edge and update tracking if degree limits are met
                assigned_edges_upto_timestamp[best_timestamp] += 1
                node_degree_remaining[u][best_timestamp] -= 1
                node_degree_remaining[v][best_timestamp] -= 1
                neg_samples.append((u, v, best_timestamp))
            else:
                pos_samples.append((u, v, None))
        else:
            pos_samples.append((u, v, None))
            
    return neg_samples, pos_samples

def timestamp_utilization_score(u, v, t, assigned_edges_upto_timestamp, node_degree_remaining):
    # Score function: prioritize timestamps with more remaining degree for u and v
    return -assigned_edges_upto_timestamp[t] + node_degree_remaining[u][t] + node_degree_remaining[v][t]


In [25]:

def get_valid_replacement_edge(original_node, possible_replacements, node_degree_remaining, 
                             assigned_edges_upto_timestamp, timestamps, max_attempts=50):
    """
    Find a valid replacement edge that maintains degree constraints with retry mechanism.
    
    Args:
        original_node: The node being replaced
        possible_replacements: List of (node1, node2) possible replacement edges
        node_degree_remaining: Dictionary tracking remaining degree capacity
        assigned_edges_upto_timestamp: Current edge assignments per timestamp
        timestamps: List of valid timestamps
        max_attempts: Maximum number of attempts to find valid replacement
    
    Returns:
        tuple: (selected_edge, is_valid) - selected edge and whether it's valid
    """
    attempts = 0
    valid_replacements = []
    
    while attempts < max_attempts and not valid_replacements:
        for edge in possible_replacements:
            node = edge[0] if edge[0] != original_node else edge[1]
            
            # Check if this replacement has any valid timestamps
            has_valid_timestamp = False
            valid_timestamps = []
            
            for t in timestamps:
                # Check both degree limits and current timestamp utilization
                if (node_degree_remaining[node][t] > 0 and 
                    node_degree_remaining[original_node][t] > 0):
                    # Consider timestamp utilization in decision
                    utilization_score = -assigned_edges_upto_timestamp[t]  # Prefer less utilized timestamps
                    valid_timestamps.append((t, utilization_score))
                    has_valid_timestamp = True
            
            if has_valid_timestamp:
                # Calculate an overall score for this replacement
                avg_utilization = sum(score for _, score in valid_timestamps) / len(valid_timestamps)
                valid_replacements.append((edge, avg_utilization))
        
        attempts += 1
        
        if not valid_replacements and attempts < max_attempts:
            # If no valid replacements found, could modify criteria and retry
            # For example, temporarily relax some constraints
            continue
    
    if not valid_replacements:
        return None, False
    
    # Sort by combined score (degree availability and utilization)
    valid_replacements.sort(
        key=lambda x: (
            sum(node_degree_remaining[x[0][0]][t] + 
                node_degree_remaining[x[0][1]][t] 
                for t in timestamps) + 
            x[1] * len(timestamps)  # Weight utilization score
        ),
        reverse=True
    )
    
    # Take top 3 choices (or all if less than 3) and randomly select from them
    top_choices = [edge for edge, _ in valid_replacements[:min(3, len(valid_replacements))]]
    return random.choice(top_choices), True

In [26]:
# V5

def assign_edges_to_timestamps(edges, timestamps, feasible_edges_dict, degree_dict, degree_limit):
    neg_samples = []
    pos_samples = []
    
    all_keys = list(feasible_edges_dict.keys())
    print(f'{len(all_keys) = }')
    
    # Initialize tracking structures
    node_degree_remaining = {node: {t: degree_limit[node] for t in timestamps} 
                             for node in degree_dict.keys()}
    assigned_edges_upto_timestamp = {t: 0 for t in timestamps}
    
    deficit_degree = deepcopy(degree_dict)
    
    # Phase 1: Initial Assignment
    print('Sorting edges', end='\r')
    edges.sort(key=lambda e: max(degree_dict[e[0]], degree_dict[e[1]]), reverse=True)
    print('Sorted edges    ')
    
    neg_samples, pos_samples = get_timestaps(edges, feasible_edges_dict, neg_samples, pos_samples, 
                                             node_degree_remaining, assigned_edges_upto_timestamp, deficit_degree)
    
    
    print(f'Generated first negative samples')
    
    for (u, v, _) in pos_samples:
        print(f'For {(u, v)} ', end=' ')
        possible_repl_u = list([(src, dst) for (src, dst) in all_keys if (u == src)])
        possible_repl_v = list([(src, dst) for (src, dst) in all_keys if (v == dst)])
        print(f'we get {len(possible_repl_u)} possible replacement of {u = }, {len(possible_repl_v)} possible replacement of {v = }')
        
        assert len(possible_repl_u) > 1, f"Not possible to find replacement for {(u, v)}"
        assert len(possible_repl_v) > 1, f"Not possible to find replacement for {(u, v)}"
        
        # Get valid replacements with max_attempts
        edge_u, valid_u = get_valid_replacement_edge(u, possible_repl_u, node_degree_remaining,
                                                    assigned_edges_upto_timestamp, timestamps,max_attempts=50  # Can adjust this value based on your needs
                                                    )
        print(f'got valid replace of {u = } {edge_u}')
        edge_v, valid_v = get_valid_replacement_edge(
                                                    v, possible_repl_v,
                                                    node_degree_remaining,
                                                    assigned_edges_upto_timestamp,
                                                    timestamps,
                                                    max_attempts=50
                                                    )
        print(f'got valid replace of {v = } {edge_v}')
        if valid_u and valid_v:
            new_neg_samples, new_pos_samples = get_timestaps(
                [edge_u, edge_v],
                feasible_edges_dict,
                neg_samples,
                pos_samples,
                node_degree_remaining,
                assigned_edges_upto_timestamp,
                deficit_degree
            )
        else:
            raise ValueError('Not found')
        
    return new_neg_samples, new_pos_samples

def get_timestaps(edges, feasible_edges_dict, neg_samples, pos_samples, node_degree_remaining, 
                  assigned_edges_upto_timestamp, deficit_degree):
    # could be a problem here, this could be a greedy approach
    for (u, v) in tqdm(edges, desc='Processing', ncols=100, leave=False):
        feasible_timestamps = feasible_edges_dict[(u, v)] 
        
        if feasible_timestamps:
            # Step 1: Filter timestamps where node degree limits aren't exceeded
            feasible_timestamps = [t for t in feasible_timestamps 
                                   if node_degree_remaining[u][t] > 0 and node_degree_remaining[v][t] > 0]
            
            if feasible_timestamps:
                # Step 2: Select the best timestamp based on utilization score
                best_timestamp = max(feasible_timestamps, key=lambda t: timestamp_utilization_score(
                    u, v, t, assigned_edges_upto_timestamp, node_degree_remaining))
                
                # Step 3: Assign edge and update tracking if degree limits are met
                assigned_edges_upto_timestamp[best_timestamp] += 1
                node_degree_remaining[u][best_timestamp] -= 1
                node_degree_remaining[v][best_timestamp] -= 1
                deficit_degree[u]-=1
                deficit_degree[v]-=1
                neg_samples.append((u, v, best_timestamp))
            else:
                pos_samples.append((u, v, None))
        else:
            pos_samples.append((u, v, None))
            
    return neg_samples, pos_samples

def timestamp_utilization_score(u, v, t, assigned_edges_upto_timestamp, node_degree_remaining):
    # Score function: prioritize timestamps with more remaining degree for u and v
    return -assigned_edges_upto_timestamp[t] + node_degree_remaining[u][t] + node_degree_remaining[v][t]


In [40]:
# V6  - b-matching solution
def assign_edges_to_timestamps(edges, timestamps, feasible_edges_dict, degree_dict, degree_limit):
    """
    Assign timestamps to edges while respecting b-matching constraints (degree limits).
    Uses a two-phase approach: initial greedy assignment followed by b-matching for failed assignments.
    
    Args:
        edges: List of (u,v) pairs representing edges
        timestamps: List of valid timestamps
        feasible_edges_dict: Dictionary mapping edges to their feasible timestamps
        degree_dict: Current degree of each node
        degree_limit: Maximum allowed degree for each node
    """
    neg_samples = []  # Successfully assigned edges with timestamps
    pos_samples = []  # Edges that need reassignment
    
    all_keys = list(feasible_edges_dict.keys())
    print(f'{len(all_keys) = }')
    
    # Initialize tracking structures for b-matching constraints
    node_degree_remaining = {node: {t: degree_limit[node] for t in timestamps} 
                           for node in degree_dict.keys()}
    assigned_edges_upto_timestamp = {t: 0 for t in timestamps}
    deficit_degree = deepcopy(degree_dict)
    
    # Phase 1: Initial Greedy Assignment
    # Sort edges by maximum degree of endpoints to handle high-degree nodes first
    print('Sorting edges', end='\r')
    edges.sort(key=lambda e: max(degree_dict[e[0]], degree_dict[e[1]]), reverse=True)
    print('Sorted edges    ')
    
    # First pass: Try to assign timestamps greedily
    neg_samples, pos_samples = get_timestaps(edges, feasible_edges_dict, neg_samples, pos_samples, 
                                           node_degree_remaining, assigned_edges_upto_timestamp, deficit_degree)
    
    print(f'Generated first negative samples')
    
    # Phase 2: B-Matching based reassignment for failed edges
    for (u, v, _) in pos_samples:
        print(f'For {(u, v)} ', end=' ')
        
        # Find potential replacement edges that satisfy b-matching constraints
        # For u: find all edges where u is source
        # For v: find all edges where v is destination
        possible_repl_u = find_b_matching_candidates(u, 'source', all_keys, node_degree_remaining, (u, v))
        possible_repl_v = find_b_matching_candidates(v, 'dest', all_keys, node_degree_remaining, (u, v))
        
        print(f'we get {len(possible_repl_u)} possible replacement of {u = }, {len(possible_repl_v)} possible replacement of {v = }')
        
        assert len(possible_repl_u) > 1, f"Not possible to find replacement for {(u, v)}"
        assert len(possible_repl_v) > 1, f"Not possible to find replacement for {(u, v)}"
        
        # Find best replacement edges that maintain b-matching constraints
        edge_u, valid_u = find_best_b_matching_edge(
            u, possible_repl_u, node_degree_remaining,
            assigned_edges_upto_timestamp, timestamps,
            max_attempts=50
        )
        print(f'got valid replace of {u = } {edge_u}')
        
        edge_v, valid_v = find_best_b_matching_edge(
            v, possible_repl_v, node_degree_remaining,
            assigned_edges_upto_timestamp, timestamps,
            max_attempts=50
        )
        print(f'got valid replace of {v = } {edge_v}')
        
        # If valid replacements found, attempt to assign timestamps
        if valid_u and valid_v:
            new_neg_samples, new_pos_samples = get_timestaps(
                [edge_u, edge_v],
                feasible_edges_dict,
                neg_samples,
                pos_samples,
                node_degree_remaining,
                assigned_edges_upto_timestamp,
                deficit_degree
            )
        else:
            raise ValueError('No valid b-matching found')
        
    return new_neg_samples, new_pos_samples

def find_b_matching_candidates(node, position, all_keys, node_degree_remaining, original_edge):
    """
    Find candidate edges for b-matching that respect degree constraints.
    
    Args:
        node: The node to find candidates for
        position: 'source' or 'dest' indicating node's position in edge
        all_keys: List of all possible edges
        node_degree_remaining: Dictionary tracking remaining degree capacity
        original_edge: The (src, dst) edge that failed and needs replacement
    """
    candidates = []
    orig_src, orig_dst = original_edge
    
    for (src, dst) in all_keys:
        # Skip the original edge that failed
        if src == orig_src and dst == orig_dst:
            continue
            
        if position == 'source' and src == node:
            # Check if adding this edge would respect b-matching constraints
            if any(node_degree_remaining[src][t] > 0 and 
                  node_degree_remaining[dst][t] > 0 
                  for t in node_degree_remaining[src].keys()):
                candidates.append((src, dst))
        elif position == 'dest' and dst == node:
            if any(node_degree_remaining[src][t] > 0 and 
                  node_degree_remaining[dst][t] > 0 
                  for t in node_degree_remaining[src].keys()):
                candidates.append((src, dst))
    return candidates

def find_best_b_matching_edge(node, candidates, node_degree_remaining, 
                            assigned_edges_upto_timestamp, timestamps, max_attempts=50):
    """
    Find the best edge from candidates that maintains b-matching constraints.
    Uses a scoring system that considers:
    1. Available degree capacity
    2. Timestamp utilization
    3. Balance of degree distribution
    
    Returns:
        tuple: (selected_edge, is_valid)
    """
    attempts = 0
    valid_candidates = []
    
    while attempts < max_attempts and not valid_candidates:
        for edge in candidates:
            src, dst = edge
            valid_timestamps = []
            
            # Check b-matching constraints across timestamps
            for t in timestamps:
                if (node_degree_remaining[src][t] > 0 and 
                    node_degree_remaining[dst][t] > 0):
                    score = compute_b_matching_score(
                        src, dst, t, 
                        node_degree_remaining,
                        assigned_edges_upto_timestamp
                    )
                    valid_timestamps.append((t, score))
            
            if valid_timestamps:
                avg_score = sum(score for _, score in valid_timestamps) / len(valid_timestamps)
                valid_candidates.append((edge, avg_score))
        
        attempts += 1
    
    if not valid_candidates:
        return None, False
        
    # Select the candidate with the best b-matching score
    valid_candidates.sort(key=lambda x: x[1], reverse=True)
    return valid_candidates[0][0], True

def compute_b_matching_score(src, dst, t, node_degree_remaining, assigned_edges_upto_timestamp):
    """
    Compute a score for an edge in b-matching context.
    Higher score indicates better candidate for b-matching.
    """
    degree_score = node_degree_remaining[src][t] + node_degree_remaining[dst][t]
    utilization_score = -assigned_edges_upto_timestamp[t]
    balance_score = min(node_degree_remaining[src][t], node_degree_remaining[dst][t])
    
    return degree_score + utilization_score + balance_score


In [41]:
# len(degree_dict.keys())
neg_samples, pos_samples = assign_edges_to_timestamps(edges=gen_edges,
                                                      timestamps=sampled_timestamps,
                                                      feasible_edges_dict=feasible_edges_dict,
                                                      degree_dict=degree_dict,
                                                      degree_limit=degree_dict
                                                      )

len(all_keys) = 250088
Sorted edges    


Generated first negative samples
For (4, 8735)  

TypeError: cannot unpack non-iterable int object

In [ ]:
len(neg_samples), len(pos_samples), len(neg_samples) + len(pos_samples), len(pos_samples)/(len(neg_samples) + len(pos_samples))

## Checks

In [ ]:
ns_df = pd.DataFrame(neg_samples, columns=['u', 'i', 'ts'])
ns_df.sort_values('ts', inplace=True)
ns_df.head()

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# Plot for source nodes (u)
ns_df_u = ns_df.sort_values(['u', 'ts'])
ns_df_u['diff'] = ns_df_u.groupby(['u'])['ts'].diff().fillna(0)
ts_diffs_u = ns_df_u['diff'].dropna()

ax1.hist(ts_diffs_u/1e3, bins=100, edgecolor='black', alpha=0.7, log=True)
ax1.set_ylabel('Count')
ax1.set_xlabel('Time Difference (s)')
ax1.set_title('Time Differences for Source Nodes (u)')

# Calculate and display statistics for u
mean_diff_u = ts_diffs_u.mean()
median_diff_u = ts_diffs_u.median()
std_diff_u = ts_diffs_u.std()

ax1.text(0.05, 0.95, f'Mean: {mean_diff_u/1e3:.2f}s\nMedian: {median_diff_u/1e3:.2f}s\nStd: {std_diff_u/1e3:.2f}s', 
         transform=ax1.transAxes, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Plot for destination nodes (i)
ns_df_i = ns_df.sort_values(['i', 'ts'])
ns_df_i['diff'] = ns_df_i.groupby(['i'])['ts'].diff().fillna(0)
ts_diffs_i = ns_df_i['diff'].dropna()

ax2.hist(ts_diffs_i/1e3, bins=100, edgecolor='black', alpha=0.7, log=True)
ax2.set_ylabel('Count')
ax2.set_xlabel('Time Difference (s)')
ax2.set_title('Time Differences for Destination Nodes (i)')

# Calculate and display statistics for i
mean_diff_i = ts_diffs_i.mean()
median_diff_i = ts_diffs_i.median()
std_diff_i = ts_diffs_i.std()

ax2.text(0.05, 0.95, f'Mean: {mean_diff_i/1e3:.2f}s\nMedian: {median_diff_i/1e3:.2f}s\nStd: {std_diff_i/1e3:.2f}s', 
         transform=ax2.transAxes, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
def check_c3_compliance(df, negative_samples, time_window=1200*1e3):
    df_sorted = df.sort_values('ts')
    compliant = []
    for u, i, ts in negative_samples:
        window_start = max(ts - time_window, df_sorted['ts'].min())
        window_df = df_sorted[(df_sorted['ts'] >= window_start) & (df_sorted['ts'] <= ts)]
        nodes_in_window = set(window_df['u']) | set(window_df['i'])
        compliant.append((u in nodes_in_window) and (i in nodes_in_window))
    return compliant

In [ ]:
compliant = check_c3_compliance(early_df, neg_samples)

plt.figure(figsize=(10, 6))
plt.plot(range(len(compliant)), compliant, 'b.')
plt.xlabel('Negative Sample Index')
plt.ylabel('C3 Compliant')
plt.title('C3 Compliance of Negative Samples')
plt.yticks([0, 1], ['No', 'Yes'])
plt.show()

print(f"Percentage of C3 compliant samples: {sum(compliant)/len(compliant)*100:.2f}%")

In [ ]:
compliant = check_c3_compliance(early_df, neg_samples+pos_samples)

plt.figure(figsize=(10, 6))
plt.plot(range(len(compliant)), compliant, 'b.')
plt.xlabel('Negative Sample Index')
plt.ylabel('C3 Compliant')
plt.title('C3 Compliance of Negative Samples')
plt.yticks([0, 1], ['No', 'Yes'])
plt.show()

print(f"Percentage of C3 compliant samples: {sum(compliant)/len(compliant)*100:.2f}%")

In [ ]:
# 100-86.55 # 13.45%

In [ ]:
sum(ns_df[['u', 'i']].value_counts() > 1)  # right any edge is added only once, we can also repeat to make up for defecit degrees.

# ns_df['u'].min(), ns_df['u'].max(), ns_df['i'].min(), ns_df['i'].max()